# Fault Subsection Demo

This notebook demonstrates the basic usage of `parserf` for exploring fault subsection data from earthquake rupture forecast models.

Topics covered:
1. Loading a `FaultModelDataset`
2. Creating `FaultSubsection` instances
3. Accessing `FaultSubsectionData` properties (name, geometry, length, depth, dip, etc.)
4. Exploring `FaultSubsectionRuptures` and participating ruptures
5. Examining the rupture data structure and available columns

## Imports

In [19]:
from parserf.models import FaultModel, FaultModelDataset
from parserf.subsection import FaultSubsection, FaultSubsectionData, FaultSubsectionRuptures

## 1. Loading a FaultModelDataset

`FaultModelDataset` is the entry point for all data access. It wraps a specific earthquake rupture forecast dataset and provides cached access to fault sections, parent IDs, and rupture scenarios.

Available fault models:
- `FaultModel.UCERF3_31` — UCERF3 fault model 3.1
- `FaultModel.UCERF3_32` — UCERF3 fault model 3.2
- `FaultModel.NSHMP_2023` — USGS NSHM CONUS v6.0.0

In [20]:
# Load the UCERF3 fault model 3.1 dataset
dataset = FaultModelDataset(FaultModel.UCERF3_31)
print("Dataset loaded:", dataset)

Dataset loaded: FaultModelDataset(model=<FaultModel.UCERF3_31: 31>)


### Explore dataset contents

The dataset exposes three main tables:
- `parent_ids` — maps fault names to their integer parent IDs
- `sections` — GeoDataFrame of all fault subsections with geometry and metadata
- `ruptures_parsed` — all rupture scenarios with parsed subsection index sets

In [21]:
# Parent fault names and IDs
print("Parent IDs shape:", dataset.parent_ids.shape)
dataset.parent_ids.head()

Parent IDs shape: (313, 3)


,name,parent_id,parent_name
0,Airport Lake,1,Airport Lake
1,Almanor,2,Almanor
2,Anacapa - Dume [1],3,Anacapa - Dume [1]
3,Anaheim,5,Anaheim
4,Antelope Valley,1000,Antelope Valley


In [22]:
# Fault subsections GeoDataFrame
print("Sections shape:", dataset.sections.shape)
print("Columns:", list(dataset.sections.columns))
dataset.sections.head()

Sections shape: (2606, 11)
Columns: ['name', 'state', 'states', 'index', 'parent-id', 'upper-depth', 'lower-depth', 'dip', 'dip-direction', 'aseismicity', 'geometry']


,name,state,states,index,parent-id,upper-depth,lower-depth,dip,dip-direction,aseismicity,geometry
0,Airport Lake (0),CA,None,0,1,0.0,13.0,50.0,89.459,0.1,"LINESTRING (-117.74953 35.74054, -117.76365 35..."
1,Airport Lake (1),CA,None,1,1,0.0,13.0,50.0,89.459,0.1,"LINESTRING (-117.76365 35.81038, -117.76492 35..."
2,Airport Lake (2),CA,None,2,1,0.0,13.0,50.0,89.459,0.1,"LINESTRING (-117.77588 35.88045, -117.78146 35..."
3,Airport Lake (3),CA,None,3,1,0.0,13.0,50.0,89.459,0.1,"LINESTRING (-117.77332 35.95033, -117.76387 35..."
4,Airport Lake (4),CA,None,4,1,0.0,13.0,50.0,89.459,0.1,"LINESTRING (-117.76413 36.02043, -117.76424 36..."


In [23]:
# Rupture scenarios
print("Ruptures shape:", dataset.ruptures_parsed.shape)
print("Columns:", list(dataset.ruptures_parsed.columns))
dataset.ruptures_parsed.head()

Ruptures shape: (253609, 8)
Columns: ['m', 'rate', 'depth', 'dip', 'width', 'rake', 'indices', 'parsed_indices']


,m,rate,depth,dip,width,rake,indices,parsed_indices
0,6.449,0.000034,1.3,50.0,15.273,-90.0,0:1,"{0, 1}"
1,6.638,0.000015,1.3,50.0,15.273,-90.0,0:2,"{0, 1, 2}"
2,6.780,0.000011,1.3,50.0,15.273,-90.0,0:3,"{0, 1, 2, 3}"
3,6.893,0.000007,1.3,50.0,15.273,-90.0,0:4,"{0, 1, 2, 3, 4}"
4,6.988,0.000040,1.3,50.0,15.273,-90.0,0:5,"{0, 1, 2, 3, 4, 5}"


## 2. Creating a FaultSubsection Instance

`FaultSubsection` is a thin facade over a single subsection. It validates the index and then exposes two sub-objects:
- `.data` — local attributes (name, geometry, depths, dip, lengths, etc.)
- `.ruptures` — rupture participation data

In [24]:
# Create a FaultSubsection for subsection index 0
sub = FaultSubsection(dataset, index=0)
print(sub)

FaultSubsection(fault_model=UCERF3_31, index=0, name='Airport Lake (0)')


## 3. Accessing FaultSubsectionData Properties

`sub.data` is a `FaultSubsectionData` object with properties for all subsection attributes.

In [25]:
# Basic identification
print("Index:      ", sub.data.index)
print("Name:       ", sub.data.name)
print("Parent ID:  ", sub.data.parent_id)
print("Parent Name:", sub.data.parent_name)

Index:       0
Name:        Airport Lake (0)
Parent ID:   1
Parent Name: Airport Lake


In [26]:
# Fault geometry and dimensions
print("Geometry:     ", sub.data.geometry)
print("Length (km):  ", round(sub.data.length_km, 3))
print("Width (km):   ", round(sub.data.width_km, 3))
print("Area (km²):   ", round(sub.data.area_km2, 3))

Geometry:      LINESTRING (-117.74953 35.74054, -117.76365 35.81038)
Length (km):   7.854
Width (km):    16.97
Area (km²):    133.277


In [27]:
# Fault orientation and seismicity parameters
print("Upper depth (km):  ", sub.data.upper_depth)
print("Lower depth (km):  ", sub.data.lower_depth)
print("Dip (degrees):     ", sub.data.dip)
print("Dip direction:     ", sub.data.dip_direction)
print("Aseismicity factor:", sub.data.aseismicity)

Upper depth (km):   0.0
Lower depth (km):   13.0
Dip (degrees):      50.0
Dip direction:      89.459
Aseismicity factor: 0.1


## 4. Exploring FaultSubsectionRuptures

`sub.ruptures` is a `FaultSubsectionRuptures` object. Its main property is `participating_ruptures`, a GeoDataFrame of all rupture scenarios that involve this subsection.

In [28]:
# Access the participating ruptures (lazily computed and cached)
participating = sub.ruptures.participating_ruptures
print("Number of participating ruptures:", len(participating))
print("Columns:", list(participating.columns))

Number of participating ruptures: 15
Columns: ['m', 'rate', 'depth', 'dip', 'width', 'rake', 'indices', 'parsed_indices', 'length_km', 'area_km2', 'parent_area_pcts', 'geometry']


In [29]:
# Preview the first few ruptures
participating.head()

,m,rate,depth,dip,width,rake,indices,parsed_indices,length_km,area_km2,parent_area_pcts,geometry
0,6.449,0.000034,1.3,50.0,15.273,-90.0,0:1,"{0, 1}",15.706558,266.544916,{'Airport Lake': 100.0},"LINESTRING (-117.74953 35.74054, -117.76365 35..."
1,6.638,0.000015,1.3,50.0,15.273,-90.0,0:2,"{0, 1, 2}",23.559808,399.816887,{'Airport Lake': 100.0},"LINESTRING (-117.74953 35.74054, -117.76365 35..."
2,6.780,0.000011,1.3,50.0,15.273,-90.0,0:3,"{0, 1, 2, 3}",31.412992,533.087726,{'Airport Lake': 100.0},"LINESTRING (-117.74953 35.74054, -117.76365 35..."
3,6.893,0.000007,1.3,50.0,15.273,-90.0,0:4,"{0, 1, 2, 3, 4}",39.269534,666.415568,{'Airport Lake': 100.0},"LINESTRING (-117.74953 35.74054, -117.76365 35..."
4,6.988,0.000040,1.3,50.0,15.273,-90.0,0:5,"{0, 1, 2, 3, 4, 5}",47.122932,799.690047,{'Airport Lake': 100.0},"LINESTRING (-117.74953 35.74054, -117.76365 35..."


In [30]:
# Preview the last few ruptures
participating.tail()

,m,rate,depth,dip,width,rake,indices,parsed_indices,length_km,area_km2,parent_area_pcts,geometry
10,6.983,0.000025,1.3,62.5,14.154,-114.5,3:0-1127:1125,"{0, 1, 2, 3, 1125, 1126, 1127}",50.147307,776.633824,"{'Airport Lake': 68.64080717789749, 'Little La...","MULTILINESTRING ((-117.74953 35.74054, -117.76..."
11,7.013,0.000005,1.3,57.8,14.574,-103.7,4:0-1127:1126,"{0, 1, 2, 3, 4, 1126, 1127}",51.758908,828.777423,"{'Airport Lake': 80.40947416063507, 'Little La...","MULTILINESTRING ((-117.74953 35.74054, -117.76..."
12,7.064,0.000012,1.3,60.7,14.318,-110.1,4:0-1127:1125,"{0, 1, 2, 3, 4, 1125, 1126, 1127}",58.003849,909.961667,"{'Airport Lake': 73.23556502454603, 'Little La...","MULTILINESTRING ((-117.74953 35.74054, -117.76..."
13,7.090,0.000022,1.3,56.7,14.671,-101.5,5:0-1127:1126,"{0, 1, 2, 3, 4, 5, 1126, 1127}",59.612306,962.051902,"{'Airport Lake': 83.1233787968393, 'Little Lak...","MULTILINESTRING ((-117.68459 35.6526, -117.721..."
14,7.134,0.000038,1.3,59.3,14.440,-106.9,5:0-1127:1125,"{0, 1, 2, 3, 4, 5, 1125, 1126, 1127}",65.857247,1043.236145,"{'Airport Lake': 76.65474881427285, 'Little La...","MULTILINESTRING ((-117.64749 35.60516, -117.68..."


## 5. Examining the Rupture Data Structure

Each row in `participating_ruptures` represents one rupture scenario involving this subsection. Key columns include:

- `parsed_indices` — set of subsection indices involved in this rupture
- `length_km` — total geodesic surface-trace length of the rupture
- `area_km2` — total fault area of the rupture
- `parent_area_pcts` — dict mapping each parent fault name to its % contribution of the rupture area
- `geometry` — merged surface-trace geometry (MultiLineString, EPSG:4326)

In [31]:
# Examine the first rupture in detail
first_rup = participating.iloc[0]

print("Subsection indices involved:", first_rup["parsed_indices"])
print("Total rupture length (km):  ", round(first_rup["length_km"], 2))
print("Total rupture area (km²):   ", round(first_rup["area_km2"], 2))
print("Parent fault area breakdown:")
for fault, pct in first_rup["parent_area_pcts"].items():
    print(f"  {fault}: {pct:.1f}%")
print("Geometry type:", first_rup["geometry"].geom_type)

Subsection indices involved: {0, 1}
Total rupture length (km):   15.71
Total rupture area (km²):    266.54
Parent fault area breakdown:
  Airport Lake: 100.0%
Geometry type: LineString


In [32]:
# Summary statistics for ruptures involving this subsection
print("Rupture length statistics (km):")
print(participating["length_km"].describe().round(2))

Rupture length statistics (km):
count    15.00
mean     41.82
std      14.04
min      15.71
25%      32.93
50%      42.29
75%      50.95
max      65.86
Name: length_km, dtype: float64


In [33]:
# Summary statistics for rupture area
print("Rupture area statistics (km²):")
print(participating["area_km2"].describe().round(2))

Rupture area statistics (km²):
count      15.00
mean      668.41
std       221.48
min       266.54
25%       521.59
50%       666.42
75%       814.23
max      1043.24
Name: area_km2, dtype: float64


## Using FaultSubsectionData and FaultSubsectionRuptures Directly

You can also instantiate `FaultSubsectionData` and `FaultSubsectionRuptures` directly without going through `FaultSubsection`.

In [34]:
# Direct instantiation of FaultSubsectionData
data = FaultSubsectionData(dataset, index=0)
print("Name via FaultSubsectionData:", data.name)
print("Length (km):", round(data.length_km, 3))

Name via FaultSubsectionData: Airport Lake (0)
Length (km): 7.854


In [35]:
# Direct instantiation of FaultSubsectionRuptures
ruptures = FaultSubsectionRuptures(dataset, index=0)
print("Participating ruptures count:", len(ruptures.participating_ruptures))

Participating ruptures count: 15
